In [12]:
import os
os.environ["KERAS_BACKEND"] = "torch" 
from functools import partial
from molexpress import layers
from molexpress.datasets import featurizers
from molexpress.datasets import encoders
from molexpress.ops.chem_ops import get_molecule
import torch
import pandas as pd 
from tqdm import tqdm

In [13]:

class GraphNeuralNetwork(torch.nn.Module):
    
    def __init__(self, dim):
        super().__init__()
        self.gcn1 = layers.GINConv(dim)
        self.gcn2 = layers.GINConv(dim)
        self.gcn3 = layers.GINConv(dim)
        self.gcn4 = layers.GINConv(dim)
        
    def forward(self, x):
        x = self.gcn1(x)
        x = self.gcn2(x)
        x = self.gcn3(x)
        x = self.gcn4(x)
        return x


class NodePrediction(torch.nn.Module):
    
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear1 = torch.nn.Linear(input_dim, input_dim) 
        self.linear2 = torch.nn.Linear(input_dim, output_dim) 
        
    def forward(self, x):
        x = self.linear1(x['node_state'])
        x = torch.nn.functional.relu(x,inplace=False)
        x = self.linear2(x)
        return x


class EdgePrediction(torch.nn.Module):
    
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear1 = torch.nn.Linear(input_dim, input_dim) 
        self.linear2 = torch.nn.Linear(input_dim, output_dim)
        self.gather_incident = layers.GatherIncident()
        
    def forward(self, x):
        x = self.gather_incident(x) # We do not use edge states but incident node states.
        x = self.linear1(x)
        x = torch.nn.functional.relu(x,inplace=False)
        x = self.linear2(x)
        return x
    
class Dataset(torch.utils.data.Dataset):
    
    def __init__(self, x):
        self.x = x

    def __len__(self):
        return len(self.x)
        
    def __getitem__(self, index):
        graph = peptide_graph_encoder(self.x[index])
        return graph

atom_featurizers = [
    featurizers.AtomType({'C', 'N', 'P', 'H', 'S', 'O'}),
    featurizers.Hybridization(),
]

bond_featurizers = [
    featurizers.BondType(),
    featurizers.Conjugated()
]

peptide_graph_encoder = encoders.PeptideGraphEncoder(
    atom_featurizers=atom_featurizers, 
    bond_featurizers=bond_featurizers,
    self_loops=False, # self_loops True adds one feature dim to edge state
    supports_masking=True, # supports_masking True adds one feature dim to node and edge state
)


In [28]:

data = pd.read_csv(r"C:\Users\harik\Desktop\Doctoral_Project\Molexpress\molexpress-main\molexpress\pretraining\canon_pubchem.txt",header=None,names=["smiles"])

smile_dataset = data["smiles"].apply(lambda x: [x]).to_list() 
# print(dataset)




In [45]:
torch_dataset[0]

{'node_state': array([[1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0

In [35]:
torch_dataset = Dataset(smile_dataset)

partial_collate_fn = partial(
    peptide_graph_encoder.masked_collate_fn, node_masking_rate=0.25, edge_masking_rate=0.25)

dataset = torch.utils.data.DataLoader(
    torch_dataset, batch_size=1, collate_fn=partial_collate_fn,shuffle=False)


graph_model = GraphNeuralNetwork(32).to('cuda')
node_pred_model = NodePrediction(32, 14).to('cuda')
edge_pred_model = EdgePrediction(32 * 2, 6).to('cuda')




In [46]:
for elem in dataset:
    print(elem)
    break

{'node_state': array([[1., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 1.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.]], dtype=float32), 'edge_state': array([[0., 0., 1., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0.],
       [0., 0., 1., 0., 1., 0.],
       ...,
       [1., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 1.],
       [1., 0., 0., 0., 1., 0.]], dtype=float32), 'edge_src': array([ 0,  1,  1, ..., 48, 49, 49]), 'edge_dst': array([ 1,  0,  2, ..., 43, 31, 34]), 'node_loss_weight': array([0., 1., 1., ..., 0., 0., 0.], dtype=float32), 'node_label': array([[1., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.]], dtype=float32), 'edge_loss_weight': array([0., 0., 0., ...,

In [36]:

optimizer = torch.optim.SGD(
    (
        list(graph_model.parameters()) + 
        list(node_pred_model.parameters()) + 
        list(edge_pred_model.parameters())
    ),
    lr=0.001, momentum=0.5
)
loss_fn = torch.nn.BCELoss(reduction='none') # use BCELoss if node/edge label (initial node/edge state) is multi-hot.
# loss_fn = torch.nn.CrossEntropyLoss(reduction='none') # use CrossEntropyLoss if node/edge label is one-hot.

def weighted_loss(pred, true, weight):
    log = torch.sigmoid(pred)    # Sigmoid() only with BCELoss
    # print(log.shape)
    # print(true.shape)
    loss = loss_fn(log, true)
    w_loss = loss * weight[:, None]      # weight[:, None] only with BCELoss
    # print(w_loss.shape)
    # print(torch.mean(w_loss).shape)
    # print(torch.mean(w_loss))
    return torch.mean(w_loss,)
    

In [37]:
graph = graph_model(test)
node_pred = node_pred_model(graph)
edge_pred = edge_pred_model(graph)

In [38]:
node_loss = weighted_loss(node_pred, graph['node_label'], graph['node_loss_weight'])
edge_loss = weighted_loss(edge_pred, graph['edge_label'], graph['edge_loss_weight'])

loss = node_loss + edge_loss

In [39]:
loss

tensor(19.9234, device='cuda:0', grad_fn=<AddBackward0>)

In [47]:
log_file = "training_log.txt"
epochs = 250
error_log_file = "error_log.txt"

with open(log_file, "w") as f:
    f.write("Epoch,Loss\n")  

# Training loop
for epoch in tqdm(range(epochs), total=epochs, desc="Training Progress"):
    loss_sum = 0
    optimizer.zero_grad()

    try:
        for ind, x in enumerate(dataset):

            graph = graph_model(x)
            node_pred = node_pred_model(graph)
            edge_pred = edge_pred_model(graph)

            node_loss = weighted_loss(node_pred, graph['node_label'], graph['node_loss_weight'])
            edge_loss = weighted_loss(edge_pred, graph['edge_label'], graph['edge_loss_weight'])

            loss = node_loss + edge_loss
            loss.backward()

            loss_sum += loss.item()

            optimizer.step()

        # tqdm.write(f"Epoch {epoch:<3} - Loss {loss_sum:.3f}")

    except ValueError:
        print(f"Error is here {ind}")


    with open(log_file, "a") as f:
        f.write(f"{epoch},{loss_sum}\n")


Training Progress:   0%|          | 1/250 [01:14<5:10:49, 74.90s/it]

Error is here 5941


Training Progress:   1%|          | 2/250 [02:30<5:10:11, 75.05s/it]

Error is here 5941


Training Progress:   1%|          | 3/250 [03:45<5:09:43, 75.24s/it]

Error is here 5941


Training Progress:   1%|          | 3/250 [04:15<5:51:01, 85.27s/it]


KeyboardInterrupt: 